# TD2 - Modele direct : effet de la permeabilite, de la porosite et de la conductivite thermique

Une colonne de sediments de 5 m sous une riviere est deja construite (modele synthetique de l'application `1D_Stream_aquifer_GridSearch`). **Vous n'avez pas a la construire** : vous changez trois proprietes du milieu, vous lancez Ginette, et vous regardez comment la temperature mesuree a 10, 20 et 30 cm sous le lit change.

| Parametre | Symbole | Variable | Valeur de reference |
|---|---|---|---|
| Permeabilite intrinseque | $\log_{10} k$ [m$^2$] | `log_k` | -12.5 |
| Porosite | $\phi$ [-] | `n` | 0.25 |
| Conductivite thermique | $\lambda$ [W/m/K] | `lam` | 2.5 |

**Mode d'emploi** : 1) executez la cellule de preparation, 2) modifiez les trois valeurs de la cellule "A vous de jouer" et executez-la, 3) recommencez avec d'autres valeurs : les courbes de tous vos essais s'accumulent sur le meme graphique.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append(str(Path.cwd().resolve().parents[1] / 'src' / 'src_python'))
from Init_folders import prepare_ginette_directories, compile_ginette
from Direct_model import (setup_ginette, initial_conditions, boundary_conditions, run_direct_model,
                          write_coordonnee_file, smooth_square_wave, configure_thermal_parameters)
td_dir = Path.cwd().resolve()
td_dir = td_dir.parent if td_dir.name == 'SYNTHETIC_CASES2' else td_dir
synth_dir = td_dir / 'SYNTHETIC_CASES2'          # dossier de travail de ce TD
prepare_ginette_directories(str(synth_dir), subdirectories=[])
compile_ginette(source='../../../src/ginette_V2.f90')
print('Dossier de travail :', synth_dir)

## Le cas synthetique (fixe)

- Colonne de 5 m (250 mailles de 2 cm), 30 jours, pas de temps de 10 min.
- Temperature en surface : cycle journalier (5 °C) + cycle hebdomadaire (3 °C) + rechauffement de 0,3 °C/jour, autour de 17 °C. Temperature imposee a la base : 10 °C.
- Charge hydraulique : 5 m a la base ; en surface, 5 m $\pm$ 0,2 m en creneaux de periode 3 semaines (alternance infiltration / exfiltration).
- Capteurs : Temp1, Temp2, Temp3 a 10, 20 et 30 cm sous la surface.

In [ ]:
# --- Cas synthetique : ne pas modifier ---
z_top, z_bottom = 0.0, -5.0
az = z_top - z_bottom
dz = az / 250
dz_obs = 0.1
dt = 600
nb_day = 30
state = 1
date_simul_bg = pd.to_datetime('2022/04/21 14:00:00')
RHO_SOLIDE, C_SOLIDE = 2650, 3500   # fixes, comme dans le modele synthetique

t = np.arange(0, nb_day * 86400, dt)
BC = pd.DataFrame({'times': t, 'days': t / 86400})
BC['T_top'] = (5 * np.sin(2 * np.pi * t / 86400) + 0.3 * BC['days']
               + 3 * np.sin(2 * np.pi * t / 604800) + 17)
BC['T_bottom'] = 10.0
BC['h_top'] = 0.2 * smooth_square_wave(t, 3 * 604800) + 5.0
BC['h_bottom'] = 5.0
BC['head_gradient'] = (BC['h_top'] - BC['h_bottom']) / az
obs_BC = pd.DataFrame({'dates': date_simul_bg + pd.to_timedelta(t, unit='s'), 'h_top': BC['h_top'],
                       'h_bottom': BC['h_bottom'], 'T_top': BC['T_top'], 'T_bottom': BC['T_bottom']})

def modele_direct(log_k, n, lam):
    # ecrit les fichiers d'entree de Ginette, lance la simulation, renvoie un tableau de temperatures
    setup_ginette(dt, state, nb_day, z_top, z_bottom, az, dz, date_simul_bg, dz_obs)
    configure_thermal_parameters(RHO_SOLIDE, C_SOLIDE)
    write_coordonnee_file(z_bottom, dz)
    initial_conditions(obs_BC, z_top, z_bottom, dz, [-5])
    boundary_conditions(obs_BC, dt)
    sim = run_direct_model(date_simul_bg, z_bottom, dz, 1, 0.0, log_k, n, lam, C_SOLIDE)
    q = pd.read_csv('Sim_velocity_profil_t.dat', sep=r'\s+', header=None, names=['Time', 'z', 'q'])
    sim.attrs['q_moy'] = q.groupby('Time')['q'].mean()   # vitesse de Darcy moyenne sur la colonne [m/s]
    return sim

essais = {}   # memoire des essais
print('Modele pret.')

### Les conditions aux limites (pour information)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
axes[0].plot(BC['days'], BC['T_top'], label='surface')
axes[0].plot(BC['days'], BC['T_bottom'], label='base (5 m)')
axes[0].set_ylabel('Temperature [°C]'); axes[0].set_title('Temperature imposee')
axes[1].plot(BC['days'], BC['h_top'] - BC['h_bottom'], color='purple')
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_ylabel('h_top - h_bottom [m]'); axes[1].set_title('Difference de charge (>0 : infiltration, <0 : exfiltration)')
for ax in axes:
    ax.set_xlabel('Temps [jours]'); ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout(); plt.show()

## A vous de jouer

Modifiez les trois valeurs ci-dessous, puis executez la cellule (la simulation prend quelques dizaines de secondes). Donnez un nom a votre essai pour le retrouver dans la legende.

Ordres de grandeur : $\log_{10} k$ entre -14 (argile) et -9 (sable grossier) ; $\phi$ entre 0.1 et 0.5 ; $\lambda$ entre 0.5 et 4 W/m/K.

In [ ]:
nom_essai = 'reference'
log_k = -12.5    # permeabilite intrinseque : log10(k [m2])
n     = 0.25     # porosite [-]
lam   = 2.5      # conductivite thermique [W/m/K]

sim = modele_direct(log_k, n, lam)
essais[nom_essai] = sim
K = 10 ** log_k * 1000 * 9.81 / 1e-3
print(f'K = {K:.2e} m/s | |q| moyen = {sim.attrs["q_moy"].abs().mean():.2e} m/s')

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), sharey=True)
for ax, col, prof in zip(axes, ['Temp1', 'Temp2', 'Temp3'], ['10', '20', '30']):
    for nom, s in essais.items():
        ax.plot(s['Time'] / 86400, s[col], label=nom, lw=2.5 if nom == nom_essai else 1.2)
    ax.set_title(f'Capteur a {prof} cm'); ax.set_xlabel('Temps [jours]'); ax.grid(alpha=0.3)
axes[0].set_ylabel('Temperature [°C]'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

Pour comparer : changez `nom_essai` et **un seul** parametre, puis re-executez la cellule precedente. Pour repartir de zero : `essais.clear()`.

## Questions

1. Diminuez puis augmentez la permeabilite ($\log_{10} k$ = -14, -12.5, -11, -10). Que devient l'amplitude du cycle journalier a 30 cm ? Le sens de l'ecoulement (creneaux de charge) devient-il visible dans les temperatures ?
2. Faites varier la conductivite thermique (0.5, 2.5, 4) avec la permeabilite de reference. Quel effet sur l'amortissement et le retard du signal ?
3. Faites varier la porosite (0.1, 0.25, 0.5). Pourquoi l'effet est-il plus faible que celui de $k$ ?
4. Quel parametre influence le plus les temperatures mesurees ? Lesquels sont difficiles a distinguer l'un de l'autre ?